In [2]:
pip install mlflow

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached alembic-1.18.5-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.2.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-3.2.1-py3-none-any.whl.metadata (5.4 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached databricks_sdk-0.120.0-py3-none-any.whl.metadata (43 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_proto-1.44.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached sqlparse-0.5.5-py3-none-any.w

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [4]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
198,"Great acting, great movie. If you are thinking...",positive
275,Tyra Banks needs to teach these girls that it'...,negative
996,I first saw this movie as a younger child. My ...,positive
396,I picked up the movie with no cover and not ev...,positive
319,"I found this movie really hard to sit through,...",negative


In [5]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [6]:
df = normalize_text(df)
df.head()

Error during text normalization: 
**********************************************************************
  Resource wordnet not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('wordnet')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load corpora/wordnet

  Searched in:
    - 'C:\\Users\\Shubham Kumar/nltk_data'
    - 'c:\\Users\\Shubham Kumar\\anaconda3\\nltk_data'
    - 'c:\\Users\\Shubham Kumar\\anaconda3\\share\\nltk_data'
    - 'c:\\Users\\Shubham Kumar\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\Shubham Kumar\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************



LookupError: 
**********************************************************************
  Resource [93mwordnet[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('wordnet')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/wordnet[0m

  Searched in:
    - 'C:\\Users\\Shubham Kumar/nltk_data'
    - 'c:\\Users\\Shubham Kumar\\anaconda3\\nltk_data'
    - 'c:\\Users\\Shubham Kumar\\anaconda3\\share\\nltk_data'
    - 'c:\\Users\\Shubham Kumar\\anaconda3\\lib\\nltk_data'
    - 'C:\\Users\\Shubham Kumar\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [7]:
df['sentiment'].value_counts()

sentiment
negative    259
positive    241
Name: count, dtype: int64

In [8]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [9]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
198,great acting great movie thinking building see...,1
275,tyra banks needs teach girls beautiful outside...,0
996,first saw movie younger child sister told thou...,1
396,picked movie cover even knowing was watched la...,1
319,found movie really hard sit through attention ...,0


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [14]:
pip install dagshub

  Using cached dagshub-0.7.1-py3-none-any.whl.metadata (12 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached gql-4.0.0-py3-none-any.whl.metadata (10 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached treelib-1.8.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pathvalidate-3.3.1-py3-none-any.whl.metadata (12 kB)
  Using cached boto3-1.43.50-py3-none-any.whl.metadata (6.6 kB)
  Using cached dagshub_annotation_converter-0.2.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached botocore-1.43.50-py3-none-any.whl.metadata (5.6 kB)
  Using cached s3transfer-0.19.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
Using cached dagshub-0.7.1-py3-none-any.whl (273 kB)
Using cached dacite-1.6.0-py3-none-any.whl (12 kB)
Using cached dagshub_

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.0 requires botocore<1.40.50,>=1.40.46, but you have botocore 1.43.50 which is incompatible.


In [15]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Kshubham0315/E_TO_E_MLOPS.mlflow')
dagshub.init(repo_owner='Kshubham0315', repo_name='E_TO_E_MLOPS.mlflow', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=ddd019fc-2bd5-4d88-8028-fb219a470983&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=959abcbb1deed0052c485287855a19d8c113d07a2032132b21f23c1ca22e79ce




Accessing as Kshubham0315

Repository E_TO_E_MLOPS.mlflow doesn't exist, creating it under current user.

Initialized MLflow to track repo "Kshubham0315/E_TO_E_MLOPS.mlflow"

Repository Kshubham0315/E_TO_E_MLOPS.mlflow initialized!

2026/07/17 16:47:08 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/d6d2f56f492345b9a4cd18bb93f60a2d', creation_time=1784287028338, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1784287028338, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [17]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-17 16:47:21,109 - INFO - Starting MLflow run...
2026-07-17 16:47:21,877 - INFO - Logging preprocessing parameters...
2026-07-17 16:47:23,070 - INFO - Initializing Logistic Regression model...
2026-07-17 16:47:23,071 - INFO - Fitting the model...
2026-07-17 16:47:23,104 - INFO - Model training complete.
2026-07-17 16:47:23,104 - INFO - Logging model parameters...
2026-07-17 16:47:23,466 - INFO - Making predictions...
2026-07-17 16:47:23,468 - INFO - Calculating evaluation metrics...
2026-07-17 16:47:23,494 - INFO - Logging evaluation metrics...
2026-07-17 16:47:25,032 - INFO - Saving and logging the model...
2026/07/17 16:47:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-17 16:48:14,318 - INFO - Model training and logging completed in 52.44 seconds.
2026-07-17 16:48:14,320 - INFO - Accuracy: 0.648
2026-07-17 16:48:14,321 - INFO - Precision: 0.631578947368421
2026-07-17 16:48:14,322 - INFO - Recall: 0.6101694915254238
2026-07-17 

🏃 View run caring-bear-773 at: https://dagshub.com/Kshubham0315/E_TO_E_MLOPS.mlflow/#/experiments/0/runs/39c418aa1d174761a349cffc7f43ad2a
🧪 View experiment at: https://dagshub.com/Kshubham0315/E_TO_E_MLOPS.mlflow/#/experiments/0
